In [1]:
import pandas as pd
import json

In [2]:
# Načtení datasetu
df = pd.read_csv("../data/AirQualityUCI.csv", sep=';', decimal=',')

# Odstranění prázdných sloupců
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Odstranění prázdných řádků
df.dropna(how='all', inplace=True)

df.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


In [3]:
# Vynechání nečíselných sloupců (Date a Time)
data_cols = df.columns[2:]

In [4]:
# Vytvoření binární masky s chybějícími hodnotami: 1 (chybějící), 0 (úplná)
binary_mask = (df[data_cols] == -200).astype(int)
binary_mask.head()

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
# Vytvoření binárních vzorů po řádcích (tuples) a vytvoření tabulky četností výskytů jednotlivých vzorů
pattern_counts = binary_mask.apply(lambda row: tuple(row), axis=1).value_counts().reset_index()

# Přejmenování sloupců výsledného DataFrame
pattern_counts.columns = ['pattern', 'count']

# Ukázka nejčastějších vzorů chybějících hodnot
pattern_counts

,pattern,count
0,"(0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)",6114
1,"(1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0)",1195
2,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)",827
3,"(1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)",428
4,"(0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0)",364
5,"(0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1)",291
6,"(0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0)",36
7,"(1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1)",31
8,"(0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1)",26
9,"(1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)",24


In [6]:
# Definice pomocné funkce pro zjištění, zda je binární vzor monotónní
def is_monotonic_pattern(pattern):
    found_one = False
    for val in pattern:
        if val == 1:
            found_one = True
        elif found_one and val == 0:
            return False
    return True

In [7]:
# Vytvoření seznamu všech vzorů, které jsou monotónní
monotonic_patterns = [
    {"pattern": list(row.pattern), "count": int(row.count)}
    for row in pattern_counts.itertuples(index=False)
    if is_monotonic_pattern(row.pattern)
]

# Výpis počtu detekovaných monotónních vzorů
print(f"Detekováno {len(monotonic_patterns)} monotónních vzorů z {len(pattern_counts)} celkem.")

# Ukázka monotónních vzorů
monotonic_patterns

Detekováno 3 monotónních vzorů z 14 celkem.


[{'pattern': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'count': 827},
 {'pattern': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'count': 31},
 {'pattern': [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'count': 12}]

In [8]:
# Uložení monotónních vzorů do souboru
with open("../missing/export_patterns/monotone_patterns.json", "w") as f:
    json.dump(monotonic_patterns, f, indent=2)

### Identifikace výskytu monotónních vzorů v časové řadě

In [9]:
# Načtení monotónních vzorů ze souboru
with open("../missing/export_patterns/monotone_patterns.json", "r") as f:
    monotonic_patterns = json.load(f)

# Filtrování: vyloučení triviálních vzorů (samé 0 nebo samé 1)
useful_patterns = [
    p["pattern"]
    for p in monotonic_patterns
    if not all(x == 0 for x in p["pattern"]) and not all(x == 1 for x in p["pattern"])
]

# Vyhledání všech řádků odpovídajících některému ze vzorů
matches = binary_mask.apply(lambda row: any(list(row) == pat for pat in useful_patterns), axis=1)

# Získání odpovídajících řádků s časovými údaji
matching_rows = df.loc[matches, ['Date', 'Time']]

# Přidání sloupce "Pattern" ke každému záznamu
matching_rows["Pattern"] = [useful_patterns[0]] * len(matching_rows)

# Výpis výsledku
matching_rows

,Date,Time,Pattern
2433,20/06/2004,03.00.00,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
2457,21/06/2004,03.00.00,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
4065,27/08/2004,03.00.00,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
4367,08/09/2004,17.00.00,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
6705,15/12/2004,03.00.00,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
6729,16/12/2004,03.00.00,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
6753,17/12/2004,03.00.00,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
7161,03/01/2005,03.00.00,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
7185,04/01/2005,03.00.00,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
8049,09/02/2005,03.00.00,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"


In [10]:
# Převod sloupců na string
matching_rows = matching_rows.copy()
matching_rows["Date"] = matching_rows["Date"].astype(str)
matching_rows["Time"] = matching_rows["Time"].astype(str)
matching_rows["Pattern"] = matching_rows["Pattern"].astype(str)

# Uložení do JSON souboru
import json
with open("../missing/export_patterns/monotone_occurrences.json", "w") as f:
    json.dump(matching_rows.to_dict(orient="records"), f, indent=2, ensure_ascii=False)